In [ ]:
# Load data & matrices from sheets
!pip -q install numpy pandas openpyxl

from google.colab import files
uploaded = files.upload()  # upload: data.xlsx dan W_S.xlsx

Saving W_S.xlsx to W_S.xlsx
Saving data.xlsx to data.xlsx


In [ ]:
#Load data

import numpy as np
import pandas as pd

DATA_PATH = "data.xlsx"
WS_PATH   = "W_S.xlsx"     # file Excel yang berisi W_T, I_T, I_S, W_S

# Nama sheet (sesuaikan bila beda)
SHEET_WS = "W_S"
SHEET_WT = "W_T"
SHEET_IS = "I_S"
SHEET_IT = "I_T"

df = pd.read_excel(DATA_PATH)

W_S = pd.read_excel(WS_PATH, sheet_name=SHEET_WS, header=None).to_numpy(dtype=float)
W_T = pd.read_excel(WS_PATH, sheet_name=SHEET_WT, header=None).to_numpy(dtype=float)
I_S = pd.read_excel(WS_PATH, sheet_name=SHEET_IS, header=None).to_numpy(dtype=float)
I_T = pd.read_excel(WS_PATH, sheet_name=SHEET_IT, header=None).to_numpy(dtype=float)

# checks
assert df.shape[0] == 175, f"Expected 175 rows, got {df.shape[0]}"
assert W_S.shape == (35,35), f"W_S must be 35x35, got {W_S.shape}"
assert W_T.shape == (5,5),   f"W_T must be 5x5, got {W_T.shape}"
assert I_S.shape == (35,35), f"I_S must be 35x35, got {I_S.shape}"
assert I_T.shape == (5,5),   f"I_T must be 5x5, got {I_T.shape}"

print("diag(W_T) from sheet:", np.diag(W_T))
print("N rows:", len(df))
print("Year counts:", df["Waktu"].value_counts().sort_index().to_dict())

# loc_id untuk FE
loc_order = df["Lokasi"].drop_duplicates().tolist()
loc_to_id = {name:i for i,name in enumerate(loc_order)}
df["loc_id"] = df["Lokasi"].map(loc_to_id)

# vektor y, lag, z
y_all    = df["Y"].astype(float).to_numpy()
y_lag    = df["Y_(t-1)"].astype(float).to_numpy()   # NaN untuk 2018 -> otomatis di-drop saat regresi
years_all= df["Waktu"].to_numpy()

# z-score global untuk Moran-2Di (N=175)
z_all = (y_all - y_all.mean()) / y_all.std(ddof=0)

n = 35
T = 5
N = 175

diag(W_T) from sheet: [1. 1. 1. 1. 1.]
N rows: 175
Year counts: {2018: 35, 2019: 35, 2020: 35, 2021: 35, 2022: 35}


In [ ]:
#CBentuk 5 konsep W_ST (pakai W_T, I_T, I_S, W_S dari sheet)

def WST_contemporary(I_T, W_S, W_T, I_S):
    return np.kron(I_T, W_S) + np.kron(W_T, I_S)

def WST_lag(W_T, W_S, I_S):
    return np.kron(W_T, W_S) + np.kron(W_T, I_S)

def WST_new1(I_T, W_S, W_T, I_S):
    return np.kron(I_T, W_S) + np.kron(W_T, I_S) + np.kron(W_T, W_S)

def WST_new2(I_T, W_S, W_T, I_S):
    return np.kron(I_T, W_S) + np.kron(W_T, I_S) + 0.5*np.kron(W_T, W_S)

def WST_new3(I_T, W_S, W_T, I_S):
    return 0.5*np.kron(I_T, W_S) + np.kron(W_T, I_S) + 0.5*np.kron(W_T, W_S)

models = {
    "Contemporary": lambda: WST_contemporary(I_T, W_S, W_T, I_S),
    "Lag":          lambda: WST_lag(W_T, W_S, I_S),
    "New-1":        lambda: WST_new1(I_T, W_S, W_T, I_S),
    "New-2":        lambda: WST_new2(I_T, W_S, W_T, I_S),
    "New-3":        lambda: WST_new3(I_T, W_S, W_T, I_S),
}

# sanity check diagonal
for name, fn in models.items():
    W_ST = fn()
    assert W_ST.shape == (N,N)
    print(name, "diag(min,max)=", np.diag(W_ST).min(), np.diag(W_ST).max(), "S0=", W_ST.sum())

Contemporary diag(min,max)= 1.0 1.0 S0= 490.0
Lag diag(min,max)= 1.0 1.0 S0= 630.0
New-1 diag(min,max)= 1.0 1.0 S0= 805.0
New-2 diag(min,max)= 1.0 1.0 S0= 647.5
New-3 diag(min,max)= 1.0 1.0 S0= 560.0


In [ ]:
#Moran-2Di + Monte Carlo within-year permutation (I_obs, p, perm_mean, CI95)

def moran_2di(z_vec: np.ndarray, W: np.ndarray) -> float:
    """
    Moran-type scaling:
    I = (N/S0) * (z' W z) / (z' z)
    """
    z = np.asarray(z_vec, dtype=float).reshape(-1,1)
    Nn = z.shape[0]
    S0 = float(W.sum())
    if S0 == 0:
        raise ValueError("S0 = sum(W) is zero.")
    num = float(z.T @ W @ z)
    den = float(z.T @ z)
    return (Nn/S0) * (num/den)

def mc_moran_within_year(z_vec, years_vec, W, n_perm=999, seed=2026):
    rng = np.random.default_rng(seed)
    z_vec = np.asarray(z_vec, dtype=float).ravel()
    years_vec = np.asarray(years_vec).ravel()

    I_obs = moran_2di(z_vec, W)

    uniq = np.unique(years_vec)
    idxs = {yr: np.where(years_vec==yr)[0] for yr in uniq}

    I_perm = np.empty(n_perm, dtype=float)
    for b in range(n_perm):
        z_perm = z_vec.copy()
        for yr, idx in idxs.items():
            z_perm[idx] = rng.permutation(z_perm[idx])
        I_perm[b] = moran_2di(z_perm, W)

    p_two = (np.sum(np.abs(I_perm) >= np.abs(I_obs)) + 1) / (n_perm + 1)
    ciL, ciH = np.percentile(I_perm, [2.5, 97.5])

    return {
        "I_obs": float(I_obs),
        "p_value": float(p_two),
        "perm_mean": float(I_perm.mean()),
        "perm_std": float(I_perm.std(ddof=1)),
        "ci95_low": float(ciL),
        "ci95_high": float(ciH),
    }

In [ ]:
#Regression Eq.(14) FEB location + Monte Carlo (block bootstrap) → RMSE mean, R²_mean + CI95


def compute_Wy_ST(y_vec, W_ST):
    return (W_ST @ y_vec.reshape(-1,1)).ravel()

def fit_eq13_FE(df_in, Wy_full):
    """
    Within (fixed effect lokasi via demeaning):
    y_it = rho*Wy_it + beta*y_{i,t-1} + mu_i + eps_it
    Meng-drop baris yang lag-nya NaN (tahun 2018).
    R2 & RMSE dihitung pada ruang within (demeaned).
    """
    d = df_in.copy()
    d["Wy"] = Wy_full

    d = d.dropna(subset=["Y_(t-1)"]).copy()  # buang 2018
    y  = d["Y"].astype(float).to_numpy()
    Wy = d["Wy"].astype(float).to_numpy()
    lag= d["Y_(t-1)"].astype(float).to_numpy()
    g  = d["loc_id"].to_numpy()

    # demean by location (within)
    y_dm  = y  - pd.Series(y).groupby(g).transform("mean").to_numpy()
    Wy_dm = Wy - pd.Series(Wy).groupby(g).transform("mean").to_numpy()
    lag_dm= lag- pd.Series(lag).groupby(g).transform("mean").to_numpy()

    X = np.column_stack([Wy_dm, lag_dm])
    coef = np.linalg.solve(X.T@X, X.T@y_dm)

    yhat = X @ coef
    resid = y_dm - yhat

    rmse = float(np.sqrt(np.mean(resid**2)))
    ss_res = float(np.sum(resid**2))
    ss_tot = float(np.sum((y_dm - y_dm.mean())**2))
    r2 = 1 - ss_res/ss_tot if ss_tot > 0 else np.nan

    return {
        "rho_hat": float(coef[0]),
        "beta_hat": float(coef[1]),
        "rmse": rmse,
        "r2": float(r2),
        "n_obs": int(len(d))
    }

def block_bootstrap_regression(df_in, Wy_full, B=499, seed=2026):
    """
    Monte Carlo via block bootstrap per lokasi.
    Menghasilkan mean + CI95 untuk RMSE dan R2 (within).
    """
    rng = np.random.default_rng(seed)
    loc_ids = np.sort(df_in["loc_id"].unique())
    nloc = len(loc_ids)

    stats = []
    for b in range(B):
        samp = rng.choice(loc_ids, size=nloc, replace=True)

        parts = []
        wy_parts = []
        for loc in samp:
            sub = df_in[df_in["loc_id"]==loc].copy()
            parts.append(sub)
            wy_parts.append(Wy_full[sub.index.to_numpy()])

        db = pd.concat(parts, axis=0).reset_index(drop=True)
        wyb = np.concatenate(wy_parts)

        try:
            out = fit_eq13_FE(db, wyb)
            stats.append(out)
        except np.linalg.LinAlgError:
            continue

    boot = pd.DataFrame(stats)
    def ci(x): return (float(np.percentile(x,2.5)), float(np.percentile(x,97.5)))

    return {
        "RMSE_mean": float(boot["rmse"].mean()),
        "RMSE_ci95_low": ci(boot["rmse"])[0],
        "RMSE_ci95_high": ci(boot["rmse"])[1],
        "R2_mean": float(boot["r2"].mean()),
        "R2_ci95_low": ci(boot["r2"])[0],
        "R2_ci95_high": ci(boot["r2"])[1],
        "B_effective": int(len(boot))
    }

In [ ]:
#Run all models (for each concepts) → full summary

results = []

for name, fn in models.items():
    W_ST = fn()
    assert W_ST.shape == (N,N), f"{name}: W_ST not 175x175"

    # --- Moran-2Di Monte Carlo (within-year) ---
    mor = mc_moran_within_year(z_all, years_all, W_ST, n_perm=999, seed=2026)

    # --- Regression Eq.13 (FE) + Monte Carlo block bootstrap ---
    Wy = compute_Wy_ST(y_all, W_ST)
    reg_point = fit_eq13_FE(df, Wy)                       # point estimates
    reg_mc = block_bootstrap_regression(df, Wy, B=499, seed=2026)  # mean+CI

    results.append({
        "Dependency_Model": name,

        # Moran-2Di
        "Moran2Di_I_obs": mor["I_obs"],
        "Moran2Di_p_value": mor["p_value"],
        "Moran2Di_perm_mean": mor["perm_mean"],
        "Moran2Di_perm_std": mor["perm_std"],
        "Moran2Di_ci95_low": mor["ci95_low"],
        "Moran2Di_ci95_high": mor["ci95_high"],

        # Regression point
        "rho_hat_point": reg_point["rho_hat"],
        "beta_hat_point": reg_point["beta_hat"],
        "RMSE_point": reg_point["rmse"],
        "R2_point": reg_point["r2"],
        "n_obs_reg": reg_point["n_obs"],

        # Regression Monte Carlo (block bootstrap)
        **reg_mc
    })

res_df = pd.DataFrame(results).sort_values("Dependency_Model").reset_index(drop=True)
res_df

/tmp/ipykernel_4122/3665698390.py:13: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  num = float(z.T @ W @ z)
/tmp/ipykernel_4122/3665698390.py:14: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  den = float(z.T @ z)
/tmp/ipykernel_4122/3665698390.py:13: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  num = float(z.T @ W @ z)
/tmp/ipykernel_4122/3665698390.py:14: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future

,Dependency_Model,Moran2Di_I_obs,Moran2Di_p_value,Moran2Di_perm_mean,Moran2Di_perm_std,Moran2Di_ci95_low,Moran2Di_ci95_high,rho_hat_point,beta_hat_point,RMSE_point,R2_point,n_obs_reg,RMSE_mean,RMSE_ci95_low,RMSE_ci95_high,R2_mean,R2_ci95_low,R2_ci95_high,B_effective
0,Contemporary,0.652903,0.001,0.414064,0.020930,0.373160,0.455242,0.576512,-0.549418,0.322138,0.898120,140,0.318521,0.276457,0.358311,0.899289,0.868772,0.927507,499
1,Lag,0.526066,0.001,0.328452,0.015874,0.297891,0.359194,0.493435,-0.749962,0.500835,0.753740,140,0.495766,0.420453,0.574221,0.756035,0.674873,0.829072,499
2,New-1,0.460161,0.001,0.286788,0.012752,0.262167,0.311238,0.340925,-0.490574,0.539947,0.713775,140,0.534365,0.445149,0.624096,0.717503,0.638726,0.795623,499
3,New-2,0.533090,0.001,0.334946,0.015791,0.304542,0.365240,0.439931,-0.531425,0.449177,0.801920,140,0.444307,0.375586,0.511367,0.804378,0.745228,0.859900,499
4,New-3,0.581557,0.001,0.365907,0.018066,0.330507,0.400983,0.548819,-0.684917,0.397187,0.845120,140,0.392927,0.335293,0.451318,0.846623,0.797012,0.892649,499


In [ ]:
#Export the results:

res_df.to_csv("summary_5_dependency_models_full.csv", index=False)

from google.colab import files
files.download("summary_5_dependency_models_full.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>